In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_squared_log_error
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [2]:
try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False

try:
    from lightgbm import LGBMRegressor
    lgbm_available = True
except:
    lgbm_available = False

try:
    from catboost import CatBoostRegressor
    cat_available = True
except:
    cat_available = False

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
def add_datetime_features(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['hour'] = df['datetime'].dt.hour
    df['weekday'] = df['datetime'].dt.weekday
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year
    return df

train = add_datetime_features(train)
test = add_datetime_features(test)

In [5]:
features = [
    "season", "holiday", "workingday", "weather",
    "temp", "atemp", "humidity", "windspeed",
    "hour", "weekday", "month", "year"
]

X = train[features]
y = train["count"]
X_test_final = test[features]

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=20, n_jobs=-1, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42
    )
}

if xgb_available:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42
    )

if lgbm_available:
    models["LightGBM"] = LGBMRegressor(
        n_estimators=300, learning_rate=0.1, random_state=42
    )

if cat_available:
    models["CatBoost"] = CatBoostRegressor(
        iterations=300, learning_rate=0.1, depth=6,
        verbose=0, random_state=42
    )

In [8]:
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    preds[preds < 0] = 0  # RMSLE requires non-negative predictions

    mae = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)
    rmsle = np.sqrt(mean_squared_log_error(y_val, preds))

    results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "RMSLE": rmsle
    }

    print(f"{name} Metrics:")
    print(f"  MAE:   {mae:.2f}")
    print(f"  RMSE:  {rmse:.2f}")
    print(f"  R2:    {r2:.4f}")
    print(f"  RMSLE: {rmsle:.4f}")


Training Linear Regression...
Linear Regression Metrics:
  MAE:   103.50
  RMSE:  140.55
  R2:    0.4015
  RMSLE: 1.3078

Training Random Forest...
Random Forest Metrics:
  MAE:   24.38
  RMSE:  39.06
  R2:    0.9538
  RMSLE: 0.3277

Training Gradient Boosting...
Gradient Boosting Metrics:
  MAE:   24.78
  RMSE:  39.00
  R2:    0.9539
  RMSLE: 0.5068

Training XGBoost...
XGBoost Metrics:
  MAE:   23.06
  RMSE:  36.30
  R2:    0.9601
  RMSLE: 0.4719

Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000188 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 274
[LightGBM] [Info] Number of data points in the train set: 8708, number of used features: 12
[LightGBM] [Info] Start training from score 191.584750
LightGBM Metrics:
  MAE:   22.95
  RMSE:  35.82
  R2:    0.9611
  RMSLE: 0.4412

Training CatBoost...
CatBoost Metrics

In [9]:
print("\n=== Model Comparison ===")
for name, metrics in results.items():
    print(f"\n{name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")


=== Model Comparison ===

Linear Regression:
  MAE: 103.5004
  RMSE: 140.5540
  R2: 0.4015
  RMSLE: 1.3078

Random Forest:
  MAE: 24.3798
  RMSE: 39.0606
  R2: 0.9538
  RMSLE: 0.3277

Gradient Boosting:
  MAE: 24.7845
  RMSE: 39.0038
  R2: 0.9539
  RMSLE: 0.5068

XGBoost:
  MAE: 23.0634
  RMSE: 36.2971
  R2: 0.9601
  RMSLE: 0.4719

LightGBM:
  MAE: 22.9537
  RMSE: 35.8221
  R2: 0.9611
  RMSLE: 0.4412

CatBoost:
  MAE: 24.7679
  RMSE: 38.4686
  R2: 0.9552
  RMSLE: 0.4989


In [10]:
# Final Model

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

from lightgbm import LGBMRegressor

In [11]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

def add_datetime_features(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['hour'] = df['datetime'].dt.hour
    df['weekday'] = df['datetime'].dt.weekday
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year
    return df

train = add_datetime_features(train)
test = add_datetime_features(test)

features = [
    "season", "holiday", "workingday", "weather",
    "temp", "atemp", "humidity", "windspeed",
    "hour", "weekday", "month", "year"
]

X = train[features]
y = train["count"]
X_test_final = test[features]

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.1,
    random_state=42
)

final_model.fit(X_train, y_train)

val_preds = final_model.predict(X_val)
val_preds[val_preds < 0] = 0

rmsle = np.sqrt(mean_squared_log_error(y_val, val_preds))
print("Final LightGBM Validation RMSLE:", rmsle)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000075 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 274
[LightGBM] [Info] Number of data points in the train set: 8708, number of used features: 12
[LightGBM] [Info] Start training from score 191.584750
Final LightGBM Validation RMSLE: 0.44119080238403496


In [13]:
final_model.fit(X, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 279
[LightGBM] [Info] Number of data points in the train set: 10886, number of used features: 12
[LightGBM] [Info] Start training from score 191.574132


LGBMRegressor(n_estimators=300, random_state=42)

In [14]:
test_preds = final_model.predict(X_test_final)
test_preds[test_preds < 0] = 0

submission = pd.DataFrame({
    "datetime": test["datetime"],
    "count": test_preds
})

submission.to_csv("submission_final_lightgbm.csv", index=False)
print("Saved submission_final_lightgbm.csv")

Saved submission_final_lightgbm.csv


In [15]:
submission

,datetime,count
0,2011-01-20 00:00:00,21.760195
1,2011-01-20 01:00:00,1.053050
2,2011-01-20 02:00:00,0.000000
3,2011-01-20 03:00:00,1.996313
4,2011-01-20 04:00:00,1.996313
...,...,...
6488,2012-12-31 19:00:00,239.952686
6489,2012-12-31 20:00:00,148.253413
6490,2012-12-31 21:00:00,119.329279
6491,2012-12-31 22:00:00,90.484105
